In [ ]:
import pandas as pd
import re
from data_gatherer.data_gatherer import DataGatherer
from data_gatherer.parser.xml_parser import XMLParser

In [ ]:
# df = pd.read_parquet("scripts/exp_input/Local_fetched_data.parquet")

dg = DataGatherer(log_level='INFO')

if dg.parser is None:
    dg.parser = XMLParser(dg.open_data_repos_ontology, dg.logger, llm_name=dg.llm)

In [ ]:
# there are some pcm ids in this csv article_ids_REV_test.csv that we want to filter the df on
article_ids = pd.read_csv("scripts/exp_input/REV_test.txt", header=None, names=['publication'])['publication'].tolist()
article_ids = [re.sub(r'https://www.ncbi.nlm.nih.gov/pmc/articles/', '', id.lower()) for id in article_ids]
len(article_ids), len(df)

In [ ]:
df_filtered = df[df['publication'].str.lower().isin([id.lower() for id in article_ids])]
df_filtered['format'].value_counts()

In [ ]:
# # Flan-t5-finetuned -- base
# ! bash k8s/run_loop.sh \
# --iterations 1 \
# --gpus 1 \
# --input article_ids_REV_test.csv \
# --max-articles-per-slice 249 \
# --output-dir k8s/output/rev_test_c1 \
# --seed-ontology data_gatherer/config/open_bio_data_repos.json \
# --job-suffix -tc1 \
# --semantic-retrieval false \
# --brute-force-regex false \
# --no-enrich

In [ ]:
# # Flan-t5-finetuned -- S3
# ! bash k8s/run_loop.sh \
#     --iterations 1 --gpus 1 \
#     --input article_ids_REV_test.csv \
#     --max-articles-per-slice 249 \
#     --output-dir k8s/output/rev_test_c2 \
#     --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#     --job-suffix -tc2 \
#     --semantic-retrieval true \
#     --top-k 3 \
#     --brute-force-regex false \
#     --no-enrich

In [ ]:
# Flan-t5-finetuned -- RS3
! bash k8s/run_loop.sh \
    --iterations 1 --gpus 1 \
    --input article_ids_REV_test.csv \
    --max-articles-per-slice 249 \
    --output-dir k8s/output/rev_test_c3 \
    --seed-ontology data_gatherer/config/open_bio_data_repos.json \
    --job-suffix -tc3 \
    --semantic-retrieval true \
    --top-k 3 \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# Flan-t5-finetuned -- R
! bash k8s/run_loop.sh \
    --iterations 1 --gpus 1 \
    --input article_ids_REV_test.csv \
    --max-articles-per-slice 249 \
    --output-dir k8s/output/rev_test_c5 \
    --seed-ontology data_gatherer/config/open_bio_data_repos.json \
    --job-suffix -tc5 \
    --semantic-retrieval false \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# # Flan-t5-finetuned -- Full Document Chunk
# ! bash k8s/run_loop.sh \
#       --iterations 1 --gpus 1 \
#       --input article_ids_REV_test.csv \
#       --max-articles-per-slice 249 \
#       --output-dir k8s/output/rev_test_c4 \
#       --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#       --job-suffix -tc4 \
#       --top-k all \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --no-enrich

In [ ]:
# from scripts.experiment_utils import compute_gpu_energy_wh

# compute_gpu_energy_wh("k8s/output/rev_test_c1/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c2/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c3/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c4/iter1/slice_1_gpu_power.csv")

In [ ]:
# # Claude Haiku 4.5 -- base 
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c1 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c2 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c3 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true \
#     --use-batch-api false

In [ ]:
# Claude Haiku 4.5 -- R
! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
    --output-dir k8s/output/rev_test_haiku_c5 \
    --model claude-haiku-4-5-20251001 --batch-size 249 \
    --brute-force-regex true --semantic-retrieval false 

In [ ]:
# # Claude Haiku 4.5 — FDR
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c4 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --full-document-read true \
#     --prompt-name CLAUDE_FDR_FewShot

In [ ]:
batch_id = 'msgbatch_01KCiZ4gBaXBMvidxusWBnQg'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='anthropic',
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_haiku_c5/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - Base
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c1b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false 

In [ ]:
batch_id = 'batch_6a51ec4742a881909cd044d78b802ca1'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c1/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c2b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false

In [ ]:
batch_id = 'batch_6a51ececf1e481908f3e5330e7918ccb'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c2/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c3b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true


In [ ]:
# gpt 5 mini - R
! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
    --output-dir k8s/output/rev_test_gpt5mini_c5 \
    --model gpt-5-mini --batch-size 249 \
    --semantic-retrieval false --brute-force-regex true

In [ ]:
batch_id = 'batch_6a58a103e25881909d2587617ac3cfbe'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c5/dataset_citations.csv')

In [ ]:
# gpt-4o-mini — FDR
! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
    --output-dir k8s/output/rev_test_gpt4omini_c4 \
    --model 'gpt-4o-mini' --batch-size 249 \
    --semantic-retrieval false --brute-force-regex false \
    --full-document-read true \
    --prompt-name GPT_FDR_FewShot

In [ ]:
batch_id = 'batch_6a6007cd2480819093e5e724389576b4'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c4/dataset_citations.csv')

In [ ]:
# # Gemini 3.5 -- base
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c1 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- S3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c2 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- RS3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c3 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex true \
#       --use-batch-api false

In [ ]:
# Gemini 3.5 -- R
!python k8s/k8s_processor.py \
      --input k8s/input/article_ids_REV_test.csv \
      --output-dir k8s/output/rev_test_gemini_c5 \
      --model gemini-3.5-flash \
      --batch-size 249 \
      --semantic-retrieval false \
      --brute-force-regex true \
      --use-batch-api false

In [ ]:
# # Gemini 3.5 -- FDR
# !python k8s/k8s_processor.py \
# --input k8s/input/article_ids_REV_test.csv \
# --output-dir k8s/output/rev_test_gemini_c4 \
# --model gemini-3.5-flash \
# --batch-size 249 \
# --full-document-read true \
# --use-batch-api false \
# --prompt-name GPT_FDR_FewShot


In [ ]:
article_ids_synapse = [re.sub(r'pmc:', '', pmcid) for pmcid in pd.read_csv("scripts/exp_input/syn66046424-20260629.csv")["pmcid"].tolist()]
article_ids_synapse

## Results

In [ ]:
! python scripts/BioDMS/eval_configs.py

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np

results = pd.read_csv("scripts/BioDMS/config_eval/results.csv")
results["model"] = results["config"].str.split(" · ").str[0]
results["variant"] = results["config"].str.extract(r"(c\d)")

models = ["T5", "GPT-5-mini", "Gemini-3.5-flash", "Haiku"]
variants = ["c1", "c2", "c3", "c4"]
variant_labels = {
    "c1": "base",
    "c2": "S3",
    "c3": "RS3",
    "c4": "FDR",
}
# model brand colors (verified against each company's actual brand assets):
#   Claude/Anthropic terracotta #D97757, OpenAI/ChatGPT teal #10A37F, Google Gemini blue #078EFA.
#   T5/Flan-T5 has no official brand color (research model, not a branded product) -> neutral gray.
model_colors = {
    "T5": "#6b7280",
    "Haiku": "#d97757",
    "GPT-5-mini": "#10a37f",
    "Gemini-3.5-flash": "#078efa",
}
metric = "gold_R"

# per-249-article cost ($) for each model x config, keyed the same as `variants`
# (c1=Base, c2=S3, c3=RS3, c4=FDR) -- plotted fine-grained (one curve per config), not summed
cost_data = {
    "T5":               {"c1": 0.0009, "c2": 0.0020, "c3": 0.0021, "c4": 0.0088},
    "Haiku":            {"c1": 1.67,   "c2": 2.09,   "c3": 2.09,   "c4": 17.71},
    "GPT-5-mini":       {"c1": 0.56,   "c2": 0.70,   "c3": 0.69,   "c4": 4.80},
    "Gemini-3.5-flash": {"c1": 2.26,   "c2": 2.79,   "c3": 2.76,   "c4": 15.90},
}

# legend labels shortened so a single-row legend fits the column width
legend_labels = {
    "T5": "T5-ft",
    "Haiku": "Haiku-4-5",
    "GPT-5-mini": "GPT-5-mini",
    "Gemini-3.5-flash": "Gemini-3.5-Flash",
}

# figsize matches the printed single-column width (~3.4in in ACM two-column layout) so
# fonts set here map ~1:1 to printed point sizes instead of being shrunk 3x on include
fig, ax = plt.subplots(figsize=(3.4, 2.5), facecolor="white", dpi=300)
ax.set_facecolor("white")

x = np.arange(len(variants))
width = 0.2
offsets = [(i - (len(models) - 1) / 2) * width for i in range(len(models))]

for i, model in enumerate(models):
    sub = results[results["model"] == model].set_index("variant")
    vals = [sub.loc[v, metric] if v in sub.index else np.nan for v in variants]
    bars = ax.bar(x + offsets[i], vals, width, label=legend_labels[model], color=model_colors[model],
                   edgecolor="white", linewidth=0.6)
    # rotated 90 deg so labels stay narrow and don't collide between adjacent bars
    ax.bar_label(bars, fmt="%.2f", padding=1, fontsize=3.5, color="#52514e")

ax.set_xticks(x)
ax.set_xticklabels([variant_labels[v] for v in variants], fontsize=6.5, color="#222")
ax.set_ylim(0, 1.1)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.tick_params(axis="y", labelsize=4, colors="#52514e")
ax.tick_params(axis="x", length=0)
# small, deliberate gap between the outermost bars and the axis frame -- not the
# full default matplotlib margin, but not flush against the frame either
edge_pad = 0.15
ax.set_xlim(x[0] + offsets[0] - width / 2 - edge_pad, x[-1] + offsets[-1] + width / 2 + edge_pad)

# Cost overlay: one short curve PER CONFIG (base/S3/RS3/FDR), each connecting only
# its own 4 model-points -- not one long line per model crossing between config
# groups, which read as messy diagonals. All curves share a single red so "red =
# cost" regardless of which config/model. Linear secondary axis -- T5's cost is
# genuinely ~3 orders of magnitude below the commercial models, so its point sits
# essentially at 0; that's the honest picture, not an artifact to hide with a log axis.
ax2 = ax.twinx()
cost_color = "#ff0000"
for k, v in enumerate(variants):
    xs = [x[k] + off for off in offsets]
    ys = [cost_data[model][v] for model in models]
    ax2.plot(xs, ys, color=cost_color, linestyle="--", marker="o", markersize=1,
              linewidth=0.5, zorder=5)

max_cost = max(v for m in cost_data.values() for v in m.values())
ax2.set_ylim(0, max_cost + 2)
ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
ax2.tick_params(axis="y", labelsize=4, colors=cost_color)
ax2.spines["right"].set_color(cost_color)
ax2.spines["top"].set_visible(False)
ax2.grid(False)

# single-row legend above the axes: reclaims the horizontal space a side legend would
# take from the bars, at the cost of a bit of vertical space (freed up by dropping the title)
handles, labels = ax.get_legend_handles_labels()
cost_proxy = Line2D([0], [0], color=cost_color, linestyle="--", marker="o", markersize=1, linewidth=0.5)
handles.append(cost_proxy)
labels.append("Cost")
ax.legend(
    handles, labels,
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=5, frameon=False,
    fontsize=6, handlelength=1.2, handletextpad=0.4, columnspacing=0.7, borderaxespad=0.1,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.spines["bottom"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("scripts/BioDMS/config_eval/gold_recall_chart.png", dpi=300,
            bbox_inches="tight", pad_inches=0, facecolor=fig.get_facecolor())
plt.show()
